In [1]:
# 210. Compute the historical mean or median per grid and hour-of-day bucket. This is the baseline NP3 could not build from a single day, and it is now possible because the pipeline has accumulated history.

import pandas as pd
import mysql.connector

conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="root",
    database="nopis"
)

query = """
SELECT
    f.grid_id,
    f.time_key,
    f.total_activity
FROM fact_network_activity f
ORDER BY f.grid_id, f.time_key
"""

activity_df = pd.read_sql_query(query, conn)

conn.close()

# Convert timestamp
activity_df["time_key"] = pd.to_datetime(
    activity_df["time_key"],
    format="%Y%m%d%H"
)

# Extract hour-of-day
activity_df["hour"] = activity_df["time_key"].dt.hour

# Calculate historical median baseline per grid and hour
baseline_df = (
    activity_df
    .groupby(["grid_id", "hour"])["total_activity"]
    .median()
    .reset_index(name="baseline_value")
)

print("Baseline rows:", len(baseline_df))
print("Unique grids:", baseline_df["grid_id"].nunique())
print("Unique hour buckets:", baseline_df["hour"].nunique())

display(baseline_df.head(10))

D:\NOPIS\tmp\ipykernel_26016\1097999496.py:22: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  activity_df = pd.read_sql_query(query, conn)


Baseline rows: 240000
Unique grids: 10000
Unique hour buckets: 24


,grid_id,hour,baseline_value
0,1,0,57.4846
1,1,1,46.3654
2,1,2,40.8574
3,1,3,35.0075
4,1,4,32.2741
5,1,5,31.2235
6,1,6,39.3498
7,1,7,60.3113
8,1,8,70.3102
9,1,9,75.1714


In [2]:
# 211. Reuse the NP3 baseline function rather than writing a second one. Generalize it to accept a bucketing key, so the within-day and hour-of-day baselines share one implementation.

def calculate_baseline(
    data,
    group_columns,
    value_column="total_activity",
    statistic="median"
):
    """
    Calculate a baseline using a configurable grouping/bucketing key.
    """

    if statistic == "median":
        baseline = (
            data
            .groupby(group_columns)[value_column]
            .median()
            .reset_index(name="baseline_value")
        )

    elif statistic == "mean":
        baseline = (
            data
            .groupby(group_columns)[value_column]
            .mean()
            .reset_index(name="baseline_value")
        )

    else:
        raise ValueError(
            "statistic must be either 'median' or 'mean'"
        )

    return baseline


# Apply the generalized function for the grid + hour-of-day baseline
baseline_df = calculate_baseline(
    activity_df,
    group_columns=["grid_id", "hour"],
    value_column="total_activity",
    statistic="median"
)

print("Baseline rows:", len(baseline_df))
print("Unique grids:", baseline_df["grid_id"].nunique())
print("Unique hour buckets:", baseline_df["hour"].nunique())

display(baseline_df.head(10))

Baseline rows: 240000
Unique grids: 10000
Unique hour buckets: 24


,grid_id,hour,baseline_value
0,1,0,57.4846
1,1,1,46.3654
2,1,2,40.8574
3,1,3,35.0075
4,1,4,32.2741
5,1,5,31.2235
6,1,6,39.3498
7,1,7,60.3113
8,1,8,70.3102
9,1,9,75.1714


In [3]:
# Check which grids exist in dim_grid but have no activity data

grid_query = """
SELECT grid_id
FROM dim_grid
"""

grid_df = pd.read_sql(grid_query, engine)

activity_grids = set(df["grid_id"].unique())
all_grids = set(grid_df["grid_id"].unique())

missing_grids = sorted(all_grids - activity_grids)

print("Total grids in dim_grid:", len(all_grids))
print("Grids in fact_network_activity:", len(activity_grids))
print("Missing grids:", len(missing_grids))
print("Missing grid IDs:", missing_grids)

Total grids in dim_grid: 10000
Grids in fact_network_activity: 9998
Missing grids: 2
Missing grid IDs: [np.int64(5239), np.int64(5339)]


In [3]:
# 212. Compute the deviation from the baseline.

# Add the hour-of-day bucket to the activity data
activity_df["hour"] = activity_df["time_key"].dt.hour

# Merge the historical baseline
anomaly_df = activity_df.merge(
    baseline_df,
    on=["grid_id", "hour"],
    how="left"
)

# Calculate deviation from historical baseline
anomaly_df["deviation"] = (
    anomaly_df["total_activity"]
    - anomaly_df["baseline_value"]
)

print("Anomaly rows:", len(anomaly_df))
print("Missing baselines:", anomaly_df["baseline_value"].isna().sum())

display(
    anomaly_df[
        [
            "grid_id",
            "time_key",
            "total_activity",
            "baseline_value",
            "deviation"
        ]
    ].head(10)
)

Anomaly rows: 1679994
Missing baselines: 0


,grid_id,time_key,total_activity,baseline_value,deviation
0,1,2013-11-01 00:00:00,62.0092,57.4846,4.5246
1,1,2013-11-01 01:00:00,46.3654,46.3654,0.0000
2,1,2013-11-01 02:00:00,42.0870,40.8574,1.2296
3,1,2013-11-01 03:00:00,35.0978,35.0075,0.0903
4,1,2013-11-01 04:00:00,32.2741,32.2741,0.0000
5,1,2013-11-01 05:00:00,35.4160,31.2235,4.1925
6,1,2013-11-01 06:00:00,36.1216,39.3498,-3.2282
7,1,2013-11-01 07:00:00,44.7950,60.3113,-15.5163
8,1,2013-11-01 08:00:00,67.4200,70.3102,-2.8902
9,1,2013-11-01 09:00:00,83.7437,75.1714,8.5723


In [4]:
# 213. Define an anomaly score using a simple standardized or percentage deviation.

anomaly_df["anomaly_score"] = (
    anomaly_df["deviation"]
    / anomaly_df["baseline_value"]
) * 100

print("Anomaly score statistics:")
print(anomaly_df["anomaly_score"].describe())

display(
    anomaly_df[
        [
            "grid_id",
            "time_key",
            "total_activity",
            "baseline_value",
            "deviation",
            "anomaly_score"
        ]
    ].head(10)
)

Anomaly score statistics:
count    1.679994e+06
mean    -1.614163e+00
std      2.887278e+01
min     -9.761221e+01
25%     -1.162958e+01
50%      0.000000e+00
75%      7.995286e+00
max      5.493846e+03
Name: anomaly_score, dtype: float64


,grid_id,time_key,total_activity,baseline_value,deviation,anomaly_score
0,1,2013-11-01 00:00:00,62.0092,57.4846,4.5246,7.870978
1,1,2013-11-01 01:00:00,46.3654,46.3654,0.0000,0.000000
2,1,2013-11-01 02:00:00,42.0870,40.8574,1.2296,3.009492
3,1,2013-11-01 03:00:00,35.0978,35.0075,0.0903,0.257945
4,1,2013-11-01 04:00:00,32.2741,32.2741,0.0000,0.000000
5,1,2013-11-01 05:00:00,35.4160,31.2235,4.1925,13.427386
6,1,2013-11-01 06:00:00,36.1216,39.3498,-3.2282,-8.203854
7,1,2013-11-01 07:00:00,44.7950,60.3113,-15.5163,-25.727020
8,1,2013-11-01 08:00:00,67.4200,70.3102,-2.8902,-4.110641
9,1,2013-11-01 09:00:00,83.7437,75.1714,8.5723,11.403672


In [5]:
# 214. Flag both unusually high and unusually low activity, and keep the direction as a field.

HIGH_THRESHOLD = 50
LOW_THRESHOLD = -50

anomaly_df["direction"] = "NORMAL"

anomaly_df.loc[
    anomaly_df["anomaly_score"] >= HIGH_THRESHOLD,
    "direction"
] = "HIGH"

anomaly_df.loc[
    anomaly_df["anomaly_score"] <= LOW_THRESHOLD,
    "direction"
] = "LOW"

anomaly_df["anomaly_flag"] = (
    anomaly_df["direction"] != "NORMAL"
)

print("Anomaly counts:")
print(anomaly_df["direction"].value_counts())

print("\nAnomaly percentages:")
print(
    (anomaly_df["direction"].value_counts(normalize=True) * 100)
    .round(2)
)

print("\nTotal anomaly flags:", anomaly_df["anomaly_flag"].sum())

display(
    anomaly_df[
        [
            "grid_id",
            "time_key",
            "total_activity",
            "baseline_value",
            "deviation",
            "anomaly_score",
            "direction",
            "anomaly_flag"
        ]
    ].head(10)
)

Anomaly counts:
direction
NORMAL    1583326
LOW         61874
HIGH        34794
Name: count, dtype: int64

Anomaly percentages:
direction
NORMAL    94.25
LOW        3.68
HIGH       2.07
Name: proportion, dtype: float64

Total anomaly flags: 96668


,grid_id,time_key,total_activity,baseline_value,deviation,anomaly_score,direction,anomaly_flag
0,1,2013-11-01 00:00:00,62.0092,57.4846,4.5246,7.870978,NORMAL,False
1,1,2013-11-01 01:00:00,46.3654,46.3654,0.0000,0.000000,NORMAL,False
2,1,2013-11-01 02:00:00,42.0870,40.8574,1.2296,3.009492,NORMAL,False
3,1,2013-11-01 03:00:00,35.0978,35.0075,0.0903,0.257945,NORMAL,False
4,1,2013-11-01 04:00:00,32.2741,32.2741,0.0000,0.000000,NORMAL,False
5,1,2013-11-01 05:00:00,35.4160,31.2235,4.1925,13.427386,NORMAL,False
6,1,2013-11-01 06:00:00,36.1216,39.3498,-3.2282,-8.203854,NORMAL,False
7,1,2013-11-01 07:00:00,44.7950,60.3113,-15.5163,-25.727020,NORMAL,False
8,1,2013-11-01 08:00:00,67.4200,70.3102,-2.8902,-4.110641,NORMAL,False
9,1,2013-11-01 09:00:00,83.7437,75.1714,8.5723,11.403672,NORMAL,False


In [10]:
# 215. Compare the anomaly flag against the ML3 classifier output and the NP3 rule alerts, and explain the disagreements.

import pandas as pd
import mysql.connector
from sklearn.linear_model import LogisticRegression

# --------------------------------------------------
# 1. Load the new ML2 feature table
# --------------------------------------------------

conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="root",
    database="nopis"
)

query = """
SELECT
    grid_id,
    feature_timestamp,
    avg_activity,
    activity_growth,
    active_hours,
    peak_ratio,
    variability,
    internet_share_avg,
    next_total_activity
FROM network_feature_table
ORDER BY feature_timestamp, grid_id
"""

ml_df = pd.read_sql_query(query, conn)
conn.close()

ml_df["feature_timestamp"] = pd.to_datetime(
    ml_df["feature_timestamp"],
    format="%Y%m%d%H"
)

# --------------------------------------------------
# 2. Define ML3 features
# --------------------------------------------------

feature_columns = [
    "avg_activity",
    "activity_growth",
    "active_hours",
    "peak_ratio",
    "variability",
    "internet_share_avg"
]

# --------------------------------------------------
# 3. Chronological 6-day / 1-day split
# --------------------------------------------------

ml_df = ml_df.sort_values(
    ["feature_timestamp", "grid_id"]
).reset_index(drop=True)

unique_dates = sorted(
    ml_df["feature_timestamp"].dt.date.unique()
)

train_dates = unique_dates[:6]
test_dates = unique_dates[-1:]

train_df = ml_df[
    ml_df["feature_timestamp"].dt.date.isin(train_dates)
].copy()

test_df = ml_df[
    ml_df["feature_timestamp"].dt.date.isin(test_dates)
].copy()

# Remove incomplete rows
train_df = train_df.dropna(
    subset=feature_columns + ["next_total_activity"]
).copy()

test_df = test_df.dropna(
    subset=feature_columns + ["next_total_activity"]
).copy()

# --------------------------------------------------
# 4. Create target using training-only 95th percentile
# --------------------------------------------------

TARGET_THRESHOLD = train_df["next_total_activity"].quantile(0.95)

train_df["target"] = (
    train_df["next_total_activity"] > TARGET_THRESHOLD
).astype(int)

test_df["target"] = (
    test_df["next_total_activity"] > TARGET_THRESHOLD
).astype(int)

# --------------------------------------------------
# 5. Train ML3
# --------------------------------------------------

X_train = train_df[feature_columns]
y_train = train_df["target"]

X_test = test_df[feature_columns]
y_test = test_df["target"]

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

# --------------------------------------------------
# 6. Create ML3 comparison dataframe
# --------------------------------------------------

ml3_comparison = test_df[
    ["grid_id", "feature_timestamp"]
].copy()

ml3_comparison["ml3_prediction"] = y_pred

print("ML3 model trained successfully.")
print("Train rows:", len(train_df))
print("Test rows:", len(test_df))
print("Threshold:", TARGET_THRESHOLD)
print("ML3 predictions:", len(y_pred))

D:\NOPIS\tmp\ipykernel_26016\114584315.py:33: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  ml_df = pd.read_sql_query(query, conn)


ML3 model trained successfully.
Train rows: 1329995
Test rows: 229999
Threshold: 1893.4091300000007
ML3 predictions: 229999


In [11]:
# 215. Add the ML4 anomaly signal to the ML3 test-period data.

anomaly_comparison = anomaly_df[
    [
        "grid_id",
        "time_key",
        "anomaly_score",
        "direction",
        "anomaly_flag"
    ]
].copy()

# Match the timestamp name used by ML3
anomaly_comparison = anomaly_comparison.rename(
    columns={"time_key": "feature_timestamp"}
)

# Merge ML4 anomaly results with ML3 predictions
ml3_comparison = ml3_comparison.merge(
    anomaly_comparison,
    on=["grid_id", "feature_timestamp"],
    how="left"
)

print("Comparison rows:", len(ml3_comparison))
print(
    "Missing anomaly results:",
    ml3_comparison["anomaly_flag"].isna().sum()
)

display(ml3_comparison.head(10))

Comparison rows: 229999
Missing anomaly results: 0


,grid_id,feature_timestamp,ml3_prediction,anomaly_score,direction,anomaly_flag
0,1,2013-11-07,0,3.842420,NORMAL,False
1,2,2013-11-07,0,3.794447,NORMAL,False
2,3,2013-11-07,0,3.744213,NORMAL,False
3,4,2013-11-07,0,3.982892,NORMAL,False
4,5,2013-11-07,0,4.366218,NORMAL,False
5,6,2013-11-07,0,3.744213,NORMAL,False
6,7,2013-11-07,0,3.744213,NORMAL,False
7,8,2013-11-07,0,3.744213,NORMAL,False
8,9,2013-11-07,0,3.744213,NORMAL,False
9,10,2013-11-07,0,6.052373,NORMAL,False


In [13]:
# 215. Remove the final test row where the NP3 t+1 alert is unavailable.

ml3_comparison = ml3_comparison.dropna(
    subset=["np3_alert"]
).copy()

ml3_comparison["np3_alert"] = (
    ml3_comparison["np3_alert"].astype(int)
)

print("Final comparison rows:", len(ml3_comparison))
print("Missing NP3 results:", ml3_comparison["np3_alert"].isna().sum())

Final comparison rows: 229998
Missing NP3 results: 0


In [16]:
# 215. Compare the anomaly flag against the ML3 classifier output and the NP3 rule alerts, and explain the disagreements.

# Start with ML3 predictions
comparison_215 = test_df[
    ["grid_id", "feature_timestamp"]
].copy()

comparison_215["ml3_prediction"] = y_pred

# Add ML4 anomaly results
comparison_215 = comparison_215.merge(
    anomaly_df[
        [
            "grid_id",
            "time_key",
            "anomaly_flag",
            "direction",
            "anomaly_score"
        ]
    ].rename(
        columns={"time_key": "feature_timestamp"}
    ),
    on=["grid_id", "feature_timestamp"],
    how="left"
)

# Build NP3 alerts directly from the raw activity data
np3_df = activity_df[
    ["grid_id", "time_key", "total_activity"]
].copy()

np3_df["date"] = np3_df["time_key"].dt.date

# NP3 daily baseline
np3_df["baseline_activity"] = (
    np3_df
    .groupby(["grid_id", "date"])["total_activity"]
    .transform("median")
)

# NP3 HIGH_ACTIVITY rule
np3_df["np3_alert"] = (
    np3_df["total_activity"]
    >= np3_df["baseline_activity"] * 1.50
).astype(int)

# Activity at t+1 belongs to feature timestamp t
np3_df["feature_timestamp"] = (
    np3_df["time_key"] - pd.Timedelta(hours=1)
)

# Keep only required columns
np3_alerts = np3_df[
    ["grid_id", "feature_timestamp", "np3_alert"]
].copy()

# Add NP3 to comparison
comparison_215 = comparison_215.merge(
    np3_alerts,
    on=["grid_id", "feature_timestamp"],
    how="left"
)

# Remove final row if t+1 NP3 activity is unavailable
comparison_215 = comparison_215.dropna(
    subset=["np3_alert"]
).copy()

comparison_215["np3_alert"] = (
    comparison_215["np3_alert"].astype(int)
)

comparison_215["anomaly_flag"] = (
    comparison_215["anomaly_flag"].astype(bool)
)

print("Final comparison rows:", len(comparison_215))
print("\nColumns:")
print(comparison_215.columns.tolist())

display(comparison_215.head(10))

Final comparison rows: 229998

Columns:
['grid_id', 'feature_timestamp', 'ml3_prediction', 'anomaly_flag', 'direction', 'anomaly_score', 'np3_alert']


,grid_id,feature_timestamp,ml3_prediction,anomaly_flag,direction,anomaly_score,np3_alert
0,1,2013-11-07,0,False,NORMAL,3.842420,0
1,2,2013-11-07,0,False,NORMAL,3.794447,0
2,3,2013-11-07,0,False,NORMAL,3.744213,0
3,4,2013-11-07,0,False,NORMAL,3.982892,0
4,5,2013-11-07,0,False,NORMAL,4.366218,0
5,6,2013-11-07,0,False,NORMAL,3.744213,0
6,7,2013-11-07,0,False,NORMAL,3.744213,0
7,8,2013-11-07,0,False,NORMAL,3.744213,0
8,9,2013-11-07,0,False,NORMAL,3.744213,0
9,10,2013-11-07,0,False,NORMAL,6.052373,0


In [18]:
# 215. Characterize ML3, ML4 and NP3 agreement and disagreement.

comparison_215["combination"] = (
    "ML3=" + comparison_215["ml3_prediction"].astype(str)
    + ", ML4=" + comparison_215["anomaly_flag"].astype(int).astype(str)
    + ", NP3=" + comparison_215["np3_alert"].astype(str)
)

combination_counts = (
    comparison_215["combination"]
    .value_counts()
    .sort_index()
)

print("ML3 vs ML4 vs NP3")
print("-----------------")
print(combination_counts)

print("\nPercentages:")
print(
    (combination_counts / len(comparison_215) * 100)
    .round(2)
)

ML3 vs ML4 vs NP3
-----------------
combination
ML3=0, ML4=0, NP3=0    197716
ML3=0, ML4=0, NP3=1     15273
ML3=0, ML4=1, NP3=0      4103
ML3=0, ML4=1, NP3=1       864
ML3=1, ML4=0, NP3=0      9530
ML3=1, ML4=0, NP3=1      2288
ML3=1, ML4=1, NP3=0       100
ML3=1, ML4=1, NP3=1       124
Name: count, dtype: int64

Percentages:
combination
ML3=0, ML4=0, NP3=0    85.96
ML3=0, ML4=0, NP3=1     6.64
ML3=0, ML4=1, NP3=0     1.78
ML3=0, ML4=1, NP3=1     0.38
ML3=1, ML4=0, NP3=0     4.14
ML3=1, ML4=0, NP3=1     0.99
ML3=1, ML4=1, NP3=0     0.04
ML3=1, ML4=1, NP3=1     0.05
Name: count, dtype: float64


In [19]:
# 216. Quantify the alert overlap between ML3, ML4, and NP3.

print("ML3 alert rate:", round(comparison_215["ml3_prediction"].mean() * 100, 2), "%")
print("ML4 alert rate:", round(comparison_215["anomaly_flag"].mean() * 100, 2), "%")
print("NP3 alert rate:", round(comparison_215["np3_alert"].mean() * 100, 2), "%")

print("\nML3 + ML4 overlap:")
print(
    pd.crosstab(
        comparison_215["ml3_prediction"],
        comparison_215["anomaly_flag"],
        normalize="all"
    ).round(4) * 100
)

print("\nML3 + NP3 overlap:")
print(
    pd.crosstab(
        comparison_215["ml3_prediction"],
        comparison_215["np3_alert"],
        normalize="all"
    ).round(4) * 100
)

print("\nML4 + NP3 overlap:")
print(
    pd.crosstab(
        comparison_215["anomaly_flag"],
        comparison_215["np3_alert"],
        normalize="all"
    ).round(4) * 100
)

ML3 alert rate: 5.24 %
ML4 alert rate: 2.26 %
NP3 alert rate: 8.06 %

ML3 + ML4 overlap:
anomaly_flag    False  True 
ml3_prediction              
0               92.60   2.16
1                5.14   0.10

ML3 + NP3 overlap:
np3_alert           0     1
ml3_prediction             
0               87.75  7.02
1                4.19  1.05

ML4 + NP3 overlap:
np3_alert         0     1
anomaly_flag             
False         90.11  7.64
True           1.83  0.43


In [20]:
# 217. Analyze the ML3-only, ML4-only, and NP3-only cases.

ml3_only = comparison_215[
    (comparison_215["ml3_prediction"] == 1) &
    (comparison_215["anomaly_flag"] == False) &
    (comparison_215["np3_alert"] == 0)
]

ml4_only = comparison_215[
    (comparison_215["ml3_prediction"] == 0) &
    (comparison_215["anomaly_flag"] == True) &
    (comparison_215["np3_alert"] == 0)
]

np3_only = comparison_215[
    (comparison_215["ml3_prediction"] == 0) &
    (comparison_215["anomaly_flag"] == False) &
    (comparison_215["np3_alert"] == 1)
]

print("ML3-only cases:", len(ml3_only))
print("ML4-only cases:", len(ml4_only))
print("NP3-only cases:", len(np3_only))

print("\nAverage anomaly score:")
print("ML3-only:", round(ml3_only["anomaly_score"].mean(), 2))
print("ML4-only:", round(ml4_only["anomaly_score"].mean(), 2))
print("NP3-only:", round(np3_only["anomaly_score"].mean(), 2))

ML3-only cases: 9530
ML4-only cases: 4103
NP3-only cases: 15273

Average anomaly score:
ML3-only: 8.53
ML4-only: 33.61
NP3-only: 8.18


In [21]:
# 218. Examine the severity of ML4 anomalies.

print("ML4 anomaly score by direction:")
print(
    comparison_215.groupby("direction")["anomaly_score"]
    .agg(["count", "mean", "min", "max"])
    .round(2)
)

print("\nML4 anomaly flag counts:")
print(
    comparison_215["anomaly_flag"]
    .value_counts()
)

ML4 anomaly score by direction:
            count   mean    min     max
direction                              
HIGH         3871  86.12  50.00  669.31
LOW          1320 -66.16 -93.71  -50.00
NORMAL     224807   4.54 -49.94   50.00

ML4 anomaly flag counts:
anomaly_flag
False    224807
True       5191
Name: count, dtype: int64


In [15]:
# Store network anomaly scores in MySQL

from sqlalchemy import text

# Create the table
create_table_query = """
CREATE TABLE IF NOT EXISTS network_anomaly_scores (
    grid_id INT,
    feature_timestamp DATETIME,
    current_value DOUBLE,
    baseline_value DOUBLE,
    deviation DOUBLE,
    anomaly_score DOUBLE,
    direction VARCHAR(10),
    anomaly_flag BOOLEAN,
    reason TEXT
)
"""

with engine.connect() as conn:
    conn.execute(text(create_table_query))
    conn.commit()

# Clear existing data so rerunning the notebook
# does not create duplicate records
with engine.connect() as conn:
    conn.execute(text("TRUNCATE TABLE network_anomaly_scores"))
    conn.commit()

# Store the results
network_anomaly_scores.to_sql(
    "network_anomaly_scores",
    con=engine,
    if_exists="append",
    index=False,
    chunksize=10000,
    method="multi"
)

print("network_anomaly_scores stored successfully.")

network_anomaly_scores stored successfully.


In [16]:
# Final ML4 verification

from sqlalchemy import text

# ---------------------------------------------------------
# 1. Check stored row count
# ---------------------------------------------------------

row_count = pd.read_sql(
    "SELECT COUNT(*) AS row_count FROM network_anomaly_scores",
    engine
)

print("1. Stored rows:")
print(row_count)


# ---------------------------------------------------------
# 2. Check HIGH / LOW / NORMAL anomalies
# ---------------------------------------------------------

direction_counts = pd.read_sql(
    """
    SELECT
        direction,
        COUNT(*) AS count
    FROM network_anomaly_scores
    GROUP BY direction
    ORDER BY count DESC
    """,
    engine
)

print("\n2. Direction counts:")
display(direction_counts)


# ---------------------------------------------------------
# 3. Check anomaly flag
# ---------------------------------------------------------

flag_counts = pd.read_sql(
    """
    SELECT
        anomaly_flag,
        COUNT(*) AS count
    FROM network_anomaly_scores
    GROUP BY anomaly_flag
    """,
    engine
)

print("\n3. Anomaly flag counts:")
display(flag_counts)


# ---------------------------------------------------------
# 4. Check baseline bucket counts
# ---------------------------------------------------------

bucket_check = pd.read_sql(
    """
    SELECT
        COUNT(*) AS baseline_rows,
        COUNT(DISTINCT grid_id) AS unique_grids,
        COUNT(DISTINCT HOUR(feature_timestamp)) AS hour_buckets
    FROM network_anomaly_scores
    """,
    engine
)

print("\n4. Baseline coverage:")
display(bucket_check)


# ---------------------------------------------------------
# 5. Check human-readable reasons
# ---------------------------------------------------------

reason_check = pd.read_sql(
    """
    SELECT
        grid_id,
        feature_timestamp,
        anomaly_score,
        direction,
        reason
    FROM network_anomaly_scores
    WHERE direction IN ('HIGH', 'LOW')
    LIMIT 10
    """,
    engine
)

print("\n5. Sample explanations:")
display(reason_check)


# ---------------------------------------------------------
# 6. Final acceptance checks
# ---------------------------------------------------------

stored_rows = row_count.iloc[0]["row_count"]

has_high = (
    "HIGH" in direction_counts["direction"].values
)

has_low = (
    "LOW" in direction_counts["direction"].values
)

has_reasons = (
    reason_check["reason"].notna().all()
)

print("\n========== ML4 ACCEPTANCE CHECK ==========")

print(
    "PASS: Rows stored"
    if stored_rows > 0
    else "FAIL: No rows stored"
)

print(
    "PASS: HIGH anomalies produced"
    if has_high
    else "FAIL: No HIGH anomalies"
)

print(
    "PASS: LOW anomalies produced"
    if has_low
    else "FAIL: No LOW anomalies"
)

print(
    "PASS: Human-readable reasons stored"
    if has_reasons
    else "FAIL: Missing reasons"
)

print("\nML4 verification complete.")

1. Stored rows:
   row_count
0    1556206

2. Direction counts:


,direction,count
0,NORMAL,1465235
1,LOW,60279
2,HIGH,30692



3. Anomaly flag counts:


,anomaly_flag,count
0,0,1465235
1,1,90971



4. Baseline coverage:


,baseline_rows,unique_grids,hour_buckets
0,1556206,9998,24



5. Sample explanations:


,grid_id,feature_timestamp,anomaly_score,direction,reason
0,1,2013-11-05 16:30:00,74.050263,HIGH,Activity is 74.1% above the historical baselin...
1,2,2013-11-05 16:30:00,73.788165,HIGH,Activity is 73.8% above the historical baselin...
2,3,2013-11-05 16:30:00,73.511689,HIGH,Activity is 73.5% above the historical baselin...
3,4,2013-11-05 16:30:00,74.822358,HIGH,Activity is 74.8% above the historical baselin...
4,5,2013-11-05 16:30:00,72.628210,HIGH,Activity is 72.6% above the historical baselin...
5,6,2013-11-05 16:30:00,73.511689,HIGH,Activity is 73.5% above the historical baselin...
6,7,2013-11-05 16:30:00,73.511689,HIGH,Activity is 73.5% above the historical baselin...
7,8,2013-11-05 16:30:00,73.511689,HIGH,Activity is 73.5% above the historical baselin...
8,9,2013-11-05 16:30:00,73.511689,HIGH,Activity is 73.5% above the historical baselin...
9,14,2013-11-02 05:30:00,52.263037,HIGH,Activity is 52.3% above the historical baselin...



========== ML4 ACCEPTANCE CHECK ==========
PASS: Rows stored
PASS: HIGH anomalies produced
PASS: LOW anomalies produced
PASS: Human-readable reasons stored

ML4 verification complete.
